# **Model Trigonometri dalam Sistem Navigasi Astronomi untuk Peningkatan Akurasi Posisi Satelit**

In [4]:
import math

## Data

In [4]:
lokasi_pengamatan = ...
local_sidereal_time = ...
suhu_lingkungan = ...
tekanan_udara = ...
azimut = ...
altitude_elevasi = ...
ketinggian_orbit_satelit = ...

## Rumus Rumus

In [31]:
class Perhitungan():
	def __init__(self, local_sidereal_time, suhu_lingkungan, 
			  tekanan_udara, azimut, altitude_elevasi, ketinggian_orbit_satelit):
		self.lokasi_pengamatan = 3.5275
		self.local_sidereal_time = local_sidereal_time
		self.suhu_lingkungan = suhu_lingkungan
		self.tekanan_udara = tekanan_udara
		self.azimut = azimut
		self.altitude_elevasi = altitude_elevasi
		self.jari_jari_bumi = 6378.137
		self.ketinggian_orbit_satelit = ketinggian_orbit_satelit
		self.jari_jari_orbit_satelit = self.jari_jari_bumi + self.ketinggian_orbit_satelit

		self.kalkulasi()		



	def kalkulasi(self):
		def sin(degree):
			return math.sin(math.radians(degree))

		def cos(degree):
			return math.cos(math.radians(degree))

		def tan(degree):
			return math.tan(math.radians(degree))

		def solve_abc(a, b, c):
			d = b**2 - 4*a*c
			if d >= 0:
				sol1 = (-b + math.sqrt(d)) / 2*a
				sol2 = (-b - math.sqrt(d)) / 2*a
				return max(sol1, sol2)

		def spatial_error():
			delta_east = self.toposentrik_east_nampak - self.toposentrik_east
			delta_north = self.toposentrik_north_nampak - self.toposentrik_north
			delta_up = self.toposentrik_up_nampak - self.toposentrik_up

			print(delta_east, delta_north, delta_up)

			return math.sqrt(delta_east**2 + delta_north**2 + delta_up**2)

		self.faktor_suhu_tekanan = (self.tekanan_udara / 1010) * (283 / (273 + self.suhu_lingkungan))
		self.faktor_suhu_tekanan = round(self.faktor_suhu_tekanan, 6)
		print(f"Faktor suhu tekanan : {self.faktor_suhu_tekanan}")

		self.argumen_tan = self.altitude_elevasi + (10.3 / (self.altitude_elevasi + 5.11))
		self.argumen_tan = round(self.argumen_tan, 6)
		print(f"Argumen tan : {self.argumen_tan}")

		self.nilai_refraksi = self.faktor_suhu_tekanan * (1.02 / tan(self.argumen_tan)) / 60
		self.nilai_refraksi = round(self.nilai_refraksi, 6)
		print(f"Nilai refraksi : {self.nilai_refraksi}")

		self.a_sebenarnya = self.altitude_elevasi - self.nilai_refraksi
		self.a_sebenarnya = round(self.a_sebenarnya, 6)
		print(f"Nilai a sebanarnya : {self.a_sebenarnya}")

		self.deklinasi = math.degrees(math.asin(sin(self.lokasi_pengamatan) * sin(self.a_sebenarnya) + cos(self.lokasi_pengamatan) * cos(self.a_sebenarnya) * cos(self.azimut)))
		self.deklinasi = round(self.deklinasi,6)
		print(f"Deklinasi : {self.deklinasi}")

		self.hour_angle = math.degrees(math.acos(round((sin(self.a_sebenarnya) - sin(self.lokasi_pengamatan)*sin(self.deklinasi)) / (cos(self.lokasi_pengamatan) * cos(self.deklinasi)), 6)))
		self.hour_angle = round(360 - self.hour_angle, 6) # karena A = 45 derajat, maka H = 360 - H
		print(f"Hour angle : {self.hour_angle}")

		self.asensio_rekta = self.local_sidereal_time - self.hour_angle + 360
		print(f"Asensio rekta : {self.asensio_rekta}")

		self.derivasi_jarak_toposentrik_satelit_nampak = solve_abc(1, 2 * self.jari_jari_bumi * math.sin(math.radians(self.altitude_elevasi)), self.jari_jari_bumi**2 - self.jari_jari_orbit_satelit**2)	
		self.derivasi_jarak_toposentrik_satelit_nampak = round(self.derivasi_jarak_toposentrik_satelit_nampak, 6)
		print(f"Deviasi jarak toposentrik nampak : {self.derivasi_jarak_toposentrik_satelit_nampak}")

		self.toposentrik_east_nampak = self.derivasi_jarak_toposentrik_satelit_nampak * cos(self.altitude_elevasi) * sin(self.azimut)
		self.toposentrik_east_nampak = round(self.toposentrik_east_nampak, 6)
		print(f"Toposentrik east nampak : {self.toposentrik_east_nampak}")

		self.toposentrik_north_nampak = self.derivasi_jarak_toposentrik_satelit_nampak * cos(self.altitude_elevasi) * cos(self.azimut)
		self.toposentrik_north_nampak = round(self.toposentrik_north_nampak, 6)
		print(f"Toposentrik north nampak : {self.toposentrik_north_nampak}")

		self.toposentrik_up_nampak = self.derivasi_jarak_toposentrik_satelit_nampak * sin(self.altitude_elevasi)
		self.toposentrik_up_nampak = round(self.toposentrik_up_nampak, 6)
		print(f"Toposentrik up nampak : {self.toposentrik_up_nampak}")		




		self.derivasi_jarak_toposentrik_satelit = solve_abc(1, 2 * self.jari_jari_bumi * math.sin(math.radians(self.a_sebenarnya)), self.jari_jari_bumi**2 - self.jari_jari_orbit_satelit**2)	
		self.derivasi_jarak_toposentrik_satelit = round(self.derivasi_jarak_toposentrik_satelit, 6)
		print(f"Deviasi jarak toposentrik : {self.derivasi_jarak_toposentrik_satelit}")

		self.toposentrik_east = self.derivasi_jarak_toposentrik_satelit * cos(self.a_sebenarnya) * sin(self.azimut)
		self.toposentrik_east = round(self.toposentrik_east, 6)
		print(f"Toposentrik east : {self.toposentrik_east}")

		self.toposentrik_north = self.derivasi_jarak_toposentrik_satelit * cos(self.a_sebenarnya) * cos(self.azimut)
		self.toposentrik_north = round(self.toposentrik_north, 6)
		print(f"Toposentrik north : {self.toposentrik_north}")

		self.toposentrik_up = self.derivasi_jarak_toposentrik_satelit * sin(self.a_sebenarnya)
		self.toposentrik_up = round(self.toposentrik_up, 6)
		print(f"Toposentrik up : {self.toposentrik_up}")

		self.error = spatial_error()
		self.error = round(self.error, 6)
		print(f"Spatial error : {self.error}")
		



In [ ]:
# Simulasi
# Perhitungan(local_sidereal_time=145, 
#             suhu_lingkungan=25, 
#             tekanan_udara=1013.25, 
#             azimut=45, 
#             altitude_elevasi=15,
#             ketinggian_orbit_satelit=800)

Faktor suhu tekanan : 0.95272
Argumen tan : 15.512183
Nilai refraksi : 0.058354
Nilai a sebanarnya : 14.941646
Deklinasi : 44.995295
Hour angle : 284.95925
Asensio rekta : 220.04075
Deviasi jarak toposentrik nampak : 2032.978987
Toposentrik east nampak : 1388.550471
Toposentrik north nampak : 1388.550471
Toposentrik up nampak : 526.17368
Deviasi jarak toposentrik : 2036.44653
Toposentrik east : 1391.297704
Toposentrik north : 1391.297704
Toposentrik up : 525.067487
-2.747233000000051 -2.747233000000051 1.106192999999962
Spatial error : 4.039584


In [32]:
# 09:44:09
Perhitungan(local_sidereal_time=112.159248, 
            suhu_lingkungan=31, 
            tekanan_udara=1012, 
            azimut=76.2, 
            altitude_elevasi=49.6,
            ketinggian_orbit_satelit=581.27)

Faktor suhu tekanan : 0.932764
Argumen tan : 49.788265
Nilai refraksi : 0.013406
Nilai a sebanarnya : 49.586594
Deklinasi : 11.606795
Hour angle : 320.004507
Asensio rekta : 152.154741
Deviasi jarak toposentrik nampak : 741.4706
Toposentrik east nampak : 466.690088
Toposentrik north nampak : 114.63008
Toposentrik up nampak : 564.658266
Deviasi jarak toposentrik : 741.598734
Toposentrik east : 466.899051
Toposentrik north : 114.681406
Toposentrik up : 564.643369
-0.2089629999999829 -0.05132599999998888 0.014897000000019034
Spatial error : 0.215689


In [33]:
# 09:53:48
Perhitungan(local_sidereal_time=114.615956, 
            suhu_lingkungan=30, 
            tekanan_udara=1012, 
            azimut=75.9, 
            altitude_elevasi=51.4,
            ketinggian_orbit_satelit=489.45)

Faktor suhu tekanan : 0.935843
Argumen tan : 51.582269
Nilai refraksi : 0.012618
Nilai a sebanarnya : 51.387382
Deklinasi : 11.526255
Hour angle : 321.851089
Asensio rekta : 152.76486699999998
Deviasi jarak toposentrik nampak : 612.658442
Toposentrik east nampak : 370.70943
Toposentrik north nampak : 93.115773
Toposentrik up nampak : 478.805115
Deviasi jarak toposentrik : 612.754388
Toposentrik east : 370.86976
Toposentrik north : 93.156045
Toposentrik up : 478.795898
-0.16032999999998765 -0.04027200000000164 0.00921699999997827
Spatial error : 0.165567


In [34]:
# 09:56:51
Perhitungan(local_sidereal_time=115.342941, 
            suhu_lingkungan=30, 
            tekanan_udara=1012, 
            azimut=75.4, 
            altitude_elevasi=52.8,
            ketinggian_orbit_satelit=489.56)

Faktor suhu tekanan : 0.935843
Argumen tan : 52.977862
Nilai refraksi : 0.011998
Nilai a sebanarnya : 52.788002
Deklinasi : 11.604504
Hour angle : 323.312499
Asensio rekta : 152.030442
Deviasi jarak toposentrik nampak : 602.479766
Toposentrik east nampak : 352.496517
Toposentrik north nampak : 91.818465
Toposentrik up nampak : 479.893159
Deviasi jarak toposentrik : 602.565399
Toposentrik east : 352.643871
Toposentrik north : 91.856848
Toposentrik up : 479.885069
-0.1473540000000071 -0.03838299999999606 0.008090000000038344
Spatial error : 0.152486


In [36]:
# 09:59:53
Perhitungan(local_sidereal_time=116.1702, 
            suhu_lingkungan=30, 
            tekanan_udara=1012, 
            azimut=75.1, 
            altitude_elevasi=53.7,
            ketinggian_orbit_satelit=470.24)

Faktor suhu tekanan : 0.935843
Argumen tan : 53.87514
Nilai refraksi : 0.011612
Nilai a sebanarnya : 53.688388
Deklinasi : 11.628138
Hour angle : 324.249511
Asensio rekta : 151.920689
Deviasi jarak toposentrik nampak : 573.043677
Toposentrik east nampak : 327.842514
Toposentrik north nampak : 87.232148
Toposentrik up nampak : 461.832106
Deviasi jarak toposentrik : 573.120452
Toposentrik east : 327.976894
Toposentrik north : 87.267904
Toposentrik up : 461.825208
-0.13438000000002148 -0.03575600000000634 0.006898000000035154
Spatial error : 0.139227


In [37]:
# 11:12:08
Perhitungan(local_sidereal_time=134.252907, 
            suhu_lingkungan=31, 
            tekanan_udara=1011, 
            azimut=64.7, 
            altitude_elevasi=69.5,
            ketinggian_orbit_satelit=470.38)

Faktor suhu tekanan : 0.931843
Argumen tan : 69.638051
Nilai refraksi : 0.005879
Nilai a sebanarnya : 69.494121
Deklinasi : 11.949548
Hour angle : 341.112145
Asensio rekta : 153.140762
Deviasi jarak toposentrik nampak : 499.793791
Toposentrik east nampak : 158.242902
Toposentrik north nampak : 74.801077
Toposentrik up nampak : 468.142944
Deviasi jarak toposentrik : 499.811488
Toposentrik east : 158.291934
Toposentrik north : 74.824254
Toposentrik up : 468.141558
-0.04903200000001107 -0.02317699999998979 0.0013860000000249784
Spatial error : 0.054252


In [38]:
# 11:13:09
Perhitungan(local_sidereal_time=134.470166, 
            suhu_lingkungan=31, 
            tekanan_udara=1011, 
            azimut=63.9, 
            altitude_elevasi=69.6,
            ketinggian_orbit_satelit=470.02)

Faktor suhu tekanan : 0.931843
Argumen tan : 69.737866
Nilai refraksi : 0.005848
Nilai a sebanarnya : 69.594152
Deklinasi : 12.167391
Hour angle : 341.318366
Asensio rekta : 153.15179999999998
Deviasi jarak toposentrik nampak : 499.113089
Toposentrik east nampak : 156.236028
Toposentrik north nampak : 76.53924
Toposentrik up nampak : 467.809709
Deviasi jarak toposentrik : 499.130578
Toposentrik east : 156.284382
Toposentrik north : 76.562929
Toposentrik up : 467.808341
-0.04835399999998913 -0.0236889999999903 0.0013680000000135806
Spatial error : 0.053862
